# Task 3.2 - Text Classification Project
**Skill Set Go EduTech - AI/ML Internship - Week 3**

**Goal:** implement a TensorFlow text-classification project on a real dataset, and produce a working notebook plus
a trained model file.

**Dataset:** IMDB movie reviews (50,000 reviews, labelled positive/negative) - built into Keras, a standard
sentiment-analysis benchmark, and large enough to be a genuine project rather than a toy example.

In [ ]:
import os, json, time
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)
os.makedirs("results", exist_ok=True)
print("TensorFlow version:", tf.__version__)

## 1. Load and inspect the data
Keras ships IMDB already tokenized into integers (one integer per word, ranked by frequency). I cap the vocabulary
at the 10,000 most frequent words - rare words add little signal and mostly just slow training down.

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 250   # reviews longer than this are truncated, shorter ones are padded

(x_train_raw, y_train), (x_test_raw, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)

print("Training reviews:", len(x_train_raw), "| Test reviews:", len(x_test_raw))
print("Label balance (train):", np.bincount(y_train))   # 0 = negative, 1 = positive
lengths = [len(r) for r in x_train_raw]
print(f"Review length - min {min(lengths)}, max {max(lengths)}, mean {np.mean(lengths):.0f}, median {int(np.median(lengths))}")

In [ ]:
word_index = keras.datasets.imdb.get_word_index()
index_word = {v + 3: k for k, v in word_index.items()}   # +3 because 0,1,2 are reserved tokens
index_word.update({0: "<pad>", 1: "<start>", 2: "<unk>"})

def decode(review_ints):
    return " ".join(index_word.get(i, "?") for i in review_ints)

print("Sample review (decoded):")
print(decode(x_train_raw[0])[:400], "...")
print("\nLabel:", "positive" if y_train[0] == 1 else "negative")

## 2. Preprocessing: padding
Reviews have very different lengths, but a neural network needs fixed-size input. Every review is padded /
truncated to `MAX_LEN` = 250 tokens. Padding is added at the start (`padding='pre'`) so the most recent words -
usually closest to the actual verdict - end up at a consistent position.

In [ ]:
x_train = keras.preprocessing.sequence.pad_sequences(x_train_raw, maxlen=MAX_LEN, padding="pre", truncating="post")
x_test = keras.preprocessing.sequence.pad_sequences(x_test_raw, maxlen=MAX_LEN, padding="pre", truncating="post")

# carve out a validation set from the training data (test set stays untouched until final evaluation)
val_split = 5000
x_val, y_val = x_train[:val_split], y_train[:val_split]
x_train_final, y_train_final = x_train[val_split:], y_train[val_split:]

print("Train:", x_train_final.shape, "| Val:", x_val.shape, "| Test:", x_test.shape)

## 3. Model
An embedding layer turns each word index into a learned dense vector, a bidirectional LSTM reads the sequence in
both directions to capture context, and a small dense head makes the final positive/negative call.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(VOCAB_SIZE, 32),
    layers.Bidirectional(layers.LSTM(32, dropout=0.2, recurrent_dropout=0.2)),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## 4. Training

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

start = time.time()
history = model.fit(
    x_train_final, y_train_final,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop],
    verbose=2,
)
train_time = time.time() - start
print(f"Training time: {train_time:.1f}s | stopped at epoch {len(history.history['loss'])}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history.history["loss"], label="Train")
ax[0].plot(history.history["val_loss"], label="Validation")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].set_title("Loss per epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(history.history["accuracy"], label="Train")
ax[1].plot(history.history["val_accuracy"], label="Validation")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy"); ax[1].set_title("Accuracy per epoch"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/text_curves.png", dpi=150)
plt.show()

## 5. Final evaluation on the test set

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

from sklearn.metrics import classification_report, confusion_matrix
test_probs = model.predict(x_test, verbose=0).squeeze()
test_preds = (test_probs > 0.5).astype(int)
print(classification_report(y_test, test_preds, target_names=["negative", "positive"], digits=3))

In [ ]:
cm = confusion_matrix(y_test, test_preds)
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["negative", "positive"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["negative", "positive"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix (test set)")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im)
plt.tight_layout()
plt.savefig("results/text_confusion_matrix.png", dpi=150)
plt.show()

### Correct and incorrect predictions

In [ ]:
wrong_idx = np.where(test_preds != y_test)[0][:3]
correct_idx = np.where(test_preds == y_test)[0][:3]

for label, idxs in [("CORRECT", correct_idx), ("WRONG", wrong_idx)]:
    print(f"--- {label} ---")
    for i in idxs:
        print(f"[true={'pos' if y_test[i] else 'neg'}, pred={'pos' if test_preds[i] else 'neg'}, "
              f"prob={test_probs[i]:.2f}]", decode(x_test_raw[i])[:200], "...\n")

## 6. Save the trained model

In [ ]:
model.save("results/text_classifier.keras")
with open("results/text_preprocessing.json", "w") as f:
    json.dump({"vocab_size": VOCAB_SIZE, "max_len": MAX_LEN}, f, indent=2)

text_results = {
    "task": "3.2", "dataset": "IMDB movie reviews (25,000 train / 25,000 test, binary sentiment)",
    "model": "Embedding(32) + BiLSTM(32) + Dense(16) + Dropout",
    "vocab_size": VOCAB_SIZE, "max_len": MAX_LEN,
    "optimizer": "Adam (lr=0.001)", "batch_size": 64,
    "epochs_run": len(history.history["loss"]),
    "test_loss": round(float(test_loss), 4),
    "test_accuracy": round(float(test_acc), 4),
    "train_time_sec": round(train_time, 1),
}
with open("results/text_results.json", "w") as f:
    json.dump(text_results, f, indent=2)
text_results

## 7. Load the saved model and classify new text

In [ ]:
loaded_model = keras.models.load_model("results/text_classifier.keras")

def classify_text(text, word_index=word_index, max_len=MAX_LEN, vocab_size=VOCAB_SIZE):
    tokens = text.lower().split()
    ints = [1] + [min(word_index.get(w, 2) + 3, vocab_size - 1) for w in tokens]
    padded = keras.preprocessing.sequence.pad_sequences([ints], maxlen=max_len, padding="pre", truncating="post")
    prob = loaded_model.predict(padded, verbose=0)[0][0]
    return ("positive" if prob > 0.5 else "negative"), float(prob)

for review in [
    "This movie was absolutely wonderful, the acting and story were fantastic.",
    "Terrible film, a complete waste of time, I almost walked out.",
]:
    label, prob = classify_text(review)
    print(f"{label} ({prob:.2f}) <- \"{review}\"")